In [1]:
! pip install torch torchaudio transformers pandas scikit-learn


[notice] A new release of pip is available: 25.2 -> 26.0.1
[notice] To update, run: pip install --upgrade pip


In [2]:
!mkdir -p ~/.kaggle
!cp kaggle.json ~/.kaggle/
!chmod 600 ~/.kaggle/kaggle.json
!kaggle competitions download -c birdclef-2026
!unzip birdclef-2026.zip

Streaming output truncated to the last 5000 lines.
  inflating: train_soundscapes/BC2026_Train_5001_S02_20240401_010000.ogg  
  inflating: train_soundscapes/BC2026_Train_5002_S02_20240401_013000.ogg  
  inflating: train_soundscapes/BC2026_Train_5003_S02_20240401_180000.ogg  
  inflating: train_soundscapes/BC2026_Train_5004_S02_20240402_013000.ogg  
  inflating: train_soundscapes/BC2026_Train_5005_S02_20240402_034500.ogg  
  inflating: train_soundscapes/BC2026_Train_5006_S02_20240402_194500.ogg  
  inflating: train_soundscapes/BC2026_Train_5007_S02_20240402_223000.ogg  
  inflating: train_soundscapes/BC2026_Train_5008_S02_20240403_010000.ogg  
  inflating: train_soundscapes/BC2026_Train_5009_S02_20240403_013000.ogg  
  inflating: train_soundscapes/BC2026_Train_5010_S02_20240403_020000.ogg  
  inflating: train_soundscapes/BC2026_Train_5011_S02_20240403_024500.ogg  
  inflating: train_soundscapes/BC2026_Train_5012_S02_20240403_230000.ogg  
  inflating: train_soundscapes/BC2026_Train_5013_

In [2]:
import os
import ast
import random
import numpy as np
import pandas as pd
import torch
import torchaudio

from torch import nn
from torch.utils.data import Dataset, DataLoader
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import MultiLabelBinarizer
from sklearn.metrics import roc_auc_score

import json
import time

from tqdm.auto import tqdm

In [3]:
df = pd.read_csv("birdclef-2026/train.csv")
print(df.columns)
print(df.head())
df["primary_label"] = df["primary_label"].astype(str)
df["filename"] = df["filename"].astype(str)
df["labels"] = df["primary_label"].apply(lambda x: [x])

Index(['primary_label', 'secondary_labels', 'type', 'latitude', 'longitude',
       'scientific_name', 'common_name', 'class_name', 'inat_taxon_id',
       'author', 'license', 'rating', 'url', 'filename', 'collection'],
      dtype='str')
  primary_label secondary_labels type  latitude  longitude scientific_name  \
0       1161364               []   []  -22.7562   -46.8666    Guyalna cuta   
1       1161364               []   []  -22.7558   -46.8700    Guyalna cuta   
2       1161364               []   []  -22.7547   -46.8728    Guyalna cuta   
3       1161364               []   []  -22.7547   -46.8728    Guyalna cuta   
4       1161364               []   []  -22.7426   -46.8985    Guyalna cuta   

    common_name class_name  inat_taxon_id         author   license  rating  \
0  Guyalna cuta    Insecta        1161364  Lucas Barbosa  cc-by-nc     0.0   
1  Guyalna cuta    Insecta        1161364  Lucas Barbosa  cc-by-nc     0.0   
2  Guyalna cuta    Insecta        1161364  Lucas Barbosa 

In [4]:
SEED = 42
SR = 32000
DURATION = 5
AUDIO_LEN = SR * DURATION
BATCH_SIZE = 64
EPOCHS = 7
LR = 1e-3
NUM_WORKERS = 4
DEVICE = "cuda" if torch.cuda.is_available() else "mps" if torch.backends.mps.is_available() else "cpu"

TRAIN_CSV = "birdclef-2026/train.csv"
TRAIN_AUDIO_DIR = "birdclef-2026/train_audio"
TEST_AUDIO_DIR = "birdclef-2026/test_soundscapes"

USE_SECONDARY_LABELS = True
SUBSET = None

In [5]:
def seed_everything(seed=42):
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    torch.cuda.manual_seed_all(seed)

def parse_secondary_labels(x):
    if pd.isna(x):
        return []
    if isinstance(x, list):
        return [str(v) for v in x]
    try:
        parsed = ast.literal_eval(x)
        if isinstance(parsed, list):
            return [str(v) for v in parsed]
        return []
    except Exception:
        return []

def random_gain(waveform, min_gain=0.7, max_gain=1.2):
    gain = np.random.uniform(min_gain, max_gain)
    return waveform * gain

def add_noise(waveform, noise_level=0.003):
    noise = torch.randn_like(waveform) * noise_level
    return waveform + noise


In [6]:
seed_everything(SEED)

import gc

df = pd.read_csv(TRAIN_CSV)

df["primary_label"] = df["primary_label"].astype(str)
df["filename"] = df["filename"].astype(str)

if USE_SECONDARY_LABELS and "secondary_labels" in df.columns:
    df["secondary_labels_parsed"] = df["secondary_labels"].apply(parse_secondary_labels)
    df["labels"] = df.apply(
        lambda row: list(dict.fromkeys([row["primary_label"]] + row["secondary_labels_parsed"])),
        axis=1
    )
else:
    df["labels"] = df["primary_label"].apply(lambda x: [x])

if SUBSET is not None:
    df = df.sample(min(SUBSET, len(df)), random_state=SEED).reset_index(drop=True)

print("Train size:", len(df))
print(df[["primary_label", "filename", "labels"]].head())

mlb = MultiLabelBinarizer()
mlb.fit(df["labels"])
NUM_CLASSES = len(mlb.classes_)

print("Num classes:", NUM_CLASSES)

train_df, val_df = train_test_split(
    df,
    test_size=0.1,
    random_state=SEED,
    shuffle=True
)

train_df = train_df.reset_index(drop=True)
val_df = val_df.reset_index(drop=True)

print("Train split:", len(train_df))
print("Val split:", len(val_df))


class BirdTrainDataset(Dataset):
    def __init__(self, df, audio_dir, mlb, augment=False):
        self.df = df.reset_index(drop=True)
        self.audio_dir = audio_dir
        self.mlb = mlb
        self.augment = augment

    def load_audio(self, path):
        waveform, sr = torchaudio.load(path)

        if waveform.shape[0] > 1:
            waveform = waveform.mean(dim=0, keepdim=True)

        if sr != SR:
            waveform = torchaudio.functional.resample(waveform, sr, SR)

        waveform = waveform.squeeze(0)

        if len(waveform) > AUDIO_LEN:
            start = np.random.randint(0, len(waveform) - AUDIO_LEN + 1)
            waveform = waveform[start:start + AUDIO_LEN]
        else:
            pad = AUDIO_LEN - len(waveform)
            waveform = torch.nn.functional.pad(waveform, (0, pad))

        return waveform.float()

    def __len__(self):
        return len(self.df)

    def __getitem__(self, idx):
        row = self.df.iloc[idx]
        path = os.path.join(self.audio_dir, row["filename"])

        audio = self.load_audio(path)

        if self.augment:
            if np.random.rand() < 0.5:
                audio = random_gain(audio)
            if np.random.rand() < 0.3:
                audio = add_noise(audio)

        labels = row["labels"]
        y = self.mlb.transform([labels])[0]
        y = torch.tensor(y, dtype=torch.float32)

        return audio, y


class BirdTestDataset(Dataset):
    def __init__(self, audio_paths):
        self.audio_paths = audio_paths

    def __len__(self):
        return len(self.audio_paths)

    def __getitem__(self, idx):
        path = self.audio_paths[idx]
        waveform, sr = torchaudio.load(path)

        if waveform.shape[0] > 1:
            waveform = waveform.mean(dim=0, keepdim=True)

        if sr != SR:
            waveform = torchaudio.functional.resample(waveform, sr, SR)

        waveform = waveform.squeeze(0).float()
        return waveform, os.path.basename(path)

class ConvBlock1D(nn.Module):
    def __init__(self, in_ch, out_ch, kernel_size, stride):
        super().__init__()
        padding = kernel_size // 2
        self.block = nn.Sequential(
            nn.Conv1d(in_ch, out_ch, kernel_size=kernel_size, stride=stride, padding=padding, bias=False),
            nn.BatchNorm1d(out_ch),
            nn.SiLU(),
        )

    def forward(self, x):
        return self.block(x)


class SEBlock1D(nn.Module):
    def __init__(self, channels, reduction=8):
        super().__init__()
        hidden = max(channels // reduction, 8)
        self.pool = nn.AdaptiveAvgPool1d(1)
        self.fc = nn.Sequential(
            nn.Linear(channels, hidden),
            nn.SiLU(),
            nn.Linear(hidden, channels),
            nn.Sigmoid()
        )

    def forward(self, x):
        b, c, _ = x.shape
        w = self.pool(x).view(b, c)
        w = self.fc(w).view(b, c, 1)
        return x * w


class ResidualBlock1D(nn.Module):
    def __init__(self, channels, kernel_size=5):
        super().__init__()
        padding = kernel_size // 2
        self.conv1 = nn.Conv1d(channels, channels, kernel_size, padding=padding, bias=False)
        self.bn1 = nn.BatchNorm1d(channels)
        self.act = nn.SiLU()
        self.conv2 = nn.Conv1d(channels, channels, kernel_size, padding=padding, bias=False)
        self.bn2 = nn.BatchNorm1d(channels)
        self.se = SEBlock1D(channels)

    def forward(self, x):
        residual = x
        out = self.conv1(x)
        out = self.bn1(out)
        out = self.act(out)

        out = self.conv2(out)
        out = self.bn2(out)
        out = self.se(out)

        out = out + residual
        out = self.act(out)
        return out


class RawBirdCNN(nn.Module):
    def __init__(self, num_classes):
        super().__init__()

        self.stem = nn.Sequential(
            ConvBlock1D(1, 32, kernel_size=11, stride=4),
            ConvBlock1D(32, 64, kernel_size=9, stride=4),
            ConvBlock1D(64, 128, kernel_size=7, stride=4),
        )

        self.stage1 = nn.Sequential(
            ResidualBlock1D(128),
            nn.MaxPool1d(4),
        )

        self.stage2 = nn.Sequential(
            ConvBlock1D(128, 192, kernel_size=5, stride=2),
            ResidualBlock1D(192),
            nn.MaxPool1d(2),
        )

        self.stage3 = nn.Sequential(
            ConvBlock1D(192, 256, kernel_size=5, stride=2),
            ResidualBlock1D(256),
        )

        self.pool_avg = nn.AdaptiveAvgPool1d(1)
        self.pool_max = nn.AdaptiveMaxPool1d(1)

        self.head = nn.Sequential(
            nn.Linear(256 * 2, 256),
            nn.SiLU(),
            nn.Dropout(0.2),
            nn.Linear(256, num_classes)
        )

    def forward(self, x):
        x = x.unsqueeze(1)

        x = self.stem(x)
        x = self.stage1(x)
        x = self.stage2(x)
        x = self.stage3(x)

        x_avg = self.pool_avg(x).squeeze(-1)
        x_max = self.pool_max(x).squeeze(-1)
        x = torch.cat([x_avg, x_max], dim=1)

        logits = self.head(x)
        return logits

train_dataset = BirdTrainDataset(train_df, TRAIN_AUDIO_DIR, mlb, augment=True)
val_dataset = BirdTrainDataset(val_df, TRAIN_AUDIO_DIR, mlb, augment=False)

train_loader = DataLoader(
    train_dataset,
    batch_size=BATCH_SIZE,
    shuffle=True,
    num_workers=0, # Changed from NUM_WORKERS to 0 to prevent macOS multiprocessing issues
    pin_memory=False,
    drop_last=True
)

val_loader = DataLoader(
    val_dataset,
    batch_size=BATCH_SIZE,
    shuffle=False,
    num_workers=0,
    pin_memory=False
)

class_counts = np.zeros(NUM_CLASSES, dtype=np.float32)
for labels in train_df["labels"]:
    vec = mlb.transform([labels])[0]
    class_counts += vec

pos_weight = (len(train_df) - class_counts) / (class_counts + 1e-6)
pos_weight = np.clip(pos_weight, 1.0, 20.0)
pos_weight = torch.tensor(pos_weight, dtype=torch.float32, device=DEVICE)

criterion = nn.BCEWithLogitsLoss(pos_weight=pos_weight)


def macro_roc_auc(y_true, y_pred):
    valid_scores = []
    for i in range(y_true.shape[1]):
        if len(np.unique(y_true[:, i])) < 2:
            continue
        try:
            score = roc_auc_score(y_true[:, i], y_pred[:, i])
            valid_scores.append(score)
        except Exception:
            pass

    if len(valid_scores) == 0:
        return np.nan
    return float(np.mean(valid_scores))

model = RawBirdCNN(NUM_CLASSES).to(DEVICE)
optimizer = torch.optim.AdamW(model.parameters(), lr=LR, weight_decay=1e-4)
scheduler = torch.optim.lr_scheduler.CosineAnnealingLR(optimizer, T_max=EPOCHS)

# scaler does not natively support mps properly in older torch, but we will leave it for cuda.
scaler = torch.cuda.amp.GradScaler(enabled=(DEVICE == "cuda"))

def train_one_epoch(model, loader, optimizer, scaler):
    model.train()
    total_loss = 0.0

    for x, y in tqdm(loader, desc="Training", leave=False):
        x = x.to(DEVICE, non_blocking=False)
        y = y.to(DEVICE, non_blocking=False)

        optimizer.zero_grad(set_to_none=True)

        with torch.cuda.amp.autocast(enabled=(DEVICE == "cuda")):
            logits = model(x)
            loss = criterion(logits, y)

        if DEVICE == "cuda":
            scaler.scale(loss).backward()
            scaler.step(optimizer)
            scaler.update()
        else:
            loss.backward()
            optimizer.step()

        total_loss += loss.item()

        # --- MAC MPS MEMORY LEAK FIX ---
        del x, y, logits, loss
        if DEVICE == "mps":
            torch.mps.empty_cache()
        # -------------------------------

    return total_loss / len(loader)


@torch.no_grad()
def validate(model, loader):
    model.eval()
    total_loss = 0.0
    all_targets = []
    all_probs = []

    for x, y in tqdm(loader, desc="Validation", leave=False):
        x = x.to(DEVICE, non_blocking=False)
        y = y.to(DEVICE, non_blocking=False)

        with torch.cuda.amp.autocast(enabled=(DEVICE == "cuda")):
            logits = model(x)
            loss = criterion(logits, y)

        probs = torch.sigmoid(logits)

        total_loss += loss.item()
        all_targets.append(y.cpu().numpy())
        all_probs.append(probs.cpu().numpy())
        
        # --- MAC MPS MEMORY LEAK FIX ---
        del x, y, logits, loss, probs
        if DEVICE == "mps":
            torch.mps.empty_cache()
        # -------------------------------

    all_targets = np.concatenate(all_targets, axis=0)
    all_probs = np.concatenate(all_probs, axis=0)

    auc = macro_roc_auc(all_targets, all_probs)
    return total_loss / len(loader), auc


best_auc = -1
best_path = "raw_bird_cnn_best.pt"

for epoch in range(EPOCHS):
    train_loss = train_one_epoch(model, train_loader, optimizer, scaler)
    val_loss, val_auc = validate(model, val_loader)
    scheduler.step()

    print(
        f"Epoch {epoch+1}/{EPOCHS} | "
        f"train_loss={train_loss:.4f} | "
        f"val_loss={val_loss:.4f} | "
        f"val_auc={val_auc:.4f}"
    )

    if val_auc > best_auc:
        best_auc = val_auc
        torch.save(
            {
                "model_state_dict": model.state_dict(),
                "classes": mlb.classes_.tolist()
            },
            best_path
        )
        print("Saved best model.")
    
    # Run global garbage collection between epochs
    gc.collect()

print("Best val AUC:", best_auc)

ckpt = torch.load(best_path, map_location=DEVICE)
model.load_state_dict(ckpt["model_state_dict"])
model.eval()

@torch.no_grad()
def predict_soundscape_file(model, audio_path, threshold=None):
    waveform, sr = torchaudio.load(audio_path)

    if waveform.shape[0] > 1:
        waveform = waveform.mean(dim=0, keepdim=True)

    if sr != SR:
        waveform = torchaudio.functional.resample(waveform, sr, SR)

    waveform = waveform.squeeze(0).float()

    probs_per_chunk = []
    for start in range(0, len(waveform), AUDIO_LEN):
        chunk = waveform[start:start + AUDIO_LEN]
        if len(chunk) < AUDIO_LEN:
            chunk = torch.nn.functional.pad(chunk, (0, AUDIO_LEN - len(chunk)))

        chunk = chunk.unsqueeze(0).to(DEVICE)

        logits = model(chunk)
        probs = torch.sigmoid(logits).squeeze(0).cpu().numpy()
        probs_per_chunk.append(probs)
        
        # --- MAC MPS MEMORY LEAK FIX ---
        del chunk, logits, probs
        if DEVICE == "mps":
            torch.mps.empty_cache()
        # -------------------------------

    probs_per_chunk = np.stack(probs_per_chunk, axis=0)

    if threshold is not None:
        pred_binary = (probs_per_chunk >= threshold).astype(int)
        return probs_per_chunk, pred_binary

    return probs_per_chunk


if os.path.exists(TEST_AUDIO_DIR):
    test_files = [
        os.path.join(TEST_AUDIO_DIR, f)
        for f in os.listdir(TEST_AUDIO_DIR)
        if f.endswith(".ogg")
    ]

    if len(test_files) > 0:
        sample_test = test_files[0]
        probs = predict_soundscape_file(model, sample_test)
        print("Pred shape for one soundscape:", probs.shape)
        print("Example file:", os.path.basename(sample_test))

Train size: 35549
  primary_label                 filename     labels
0       1161364  1161364/iNat1216197.ogg  [1161364]
1       1161364  1161364/iNat1114648.ogg  [1161364]
2       1161364   1161364/iNat810195.ogg  [1161364]
3       1161364   1161364/iNat818781.ogg  [1161364]
4       1161364   1161364/iNat556514.ogg  [1161364]
Num classes: 206
Train split: 31994
Val split: 3555


/var/folders/dp/y8tgtl0x59n2jb9l03_0kjww0000gn/T/ipykernel_42027/3452375620.py:276: FutureWarning: `torch.cuda.amp.GradScaler(args...)` is deprecated. Please use `torch.amp.GradScaler('cuda', args...)` instead.
  scaler = torch.cuda.amp.GradScaler(enabled=(DEVICE == "cuda"))


Training:   0%|          | 0/499 [00:00<?, ?it/s]

/var/folders/dp/y8tgtl0x59n2jb9l03_0kjww0000gn/T/ipykernel_42027/3452375620.py:288: FutureWarning: `torch.cuda.amp.autocast(args...)` is deprecated. Please use `torch.amp.autocast('cuda', args...)` instead.
  with torch.cuda.amp.autocast(enabled=(DEVICE == "cuda")):


Validation:   0%|          | 0/56 [00:00<?, ?it/s]

...

Training:   0%|          | 0/499 [00:00<?, ?it/s]

/var/folders/dp/y8tgtl0x59n2jb9l03_0kjww0000gn/T/ipykernel_42027/3452375620.py:288: FutureWarning: `torch.cuda.amp.autocast(args...)` is deprecated. Please use `torch.amp.autocast('cuda', args...)` instead.
  with torch.cuda.amp.autocast(enabled=(DEVICE == "cuda")):


Validation:   0%|          | 0/56 [00:00<?, ?it/s]

/var/folders/dp/y8tgtl0x59n2jb9l03_0kjww0000gn/T/ipykernel_42027/3452375620.py:322: FutureWarning: `torch.cuda.amp.autocast(args...)` is deprecated. Please use `torch.amp.autocast('cuda', args...)` instead.
  with torch.cuda.amp.autocast(enabled=(DEVICE == "cuda")):


Epoch 7/7 | train_loss=0.1982 | val_loss=0.1979 | val_auc=0.8953
Saved best model.
Best val AUC: 0.8953221572876628


In [8]:
SUBSET = None
USE_SECONDARY_LABELS = False
BATCH_SIZE = 64
NUM_WORKERS = 0 # Changed from 2 to 0 for macOS
BASELINE_EPOCHS = 6
BASELINE_LR = 1e-3

PSEUDO_CONF_THRESHOLD = 0.90
PSEUDO_MIN_KEEP_PROB = 0.10
PSEUDO_BATCH_SIZE = 32


PSEUDO_RATIO = 0.25
PSEUDO_FINETUNE_EPOCHS = 4
PSEUDO_FINETUNE_LR = 5e-4


BASELINE_MODEL_PATH = "raw_bird_base_best.pt"
PSEUDO_MODEL_PATH = "raw_bird_pseudo_best.pt"
PSEUDO_JSON_PATH = "pseudo_labels.json"


seed_everything(SEED)

def macro_roc_auc(y_true, y_pred):
    scores = []
    for i in range(y_true.shape[1]):
        if len(np.unique(y_true[:, i])) < 2:
            continue
        try:
            scores.append(roc_auc_score(y_true[:, i], y_pred[:, i]))
        except Exception:
            pass
    if len(scores) == 0:
        return np.nan
    return float(np.mean(scores))

def format_seconds(sec):
    m = int(sec // 60)
    s = int(sec % 60)
    return f"{m:02d}:{s:02d}"


df = pd.read_csv(TRAIN_CSV)
df["primary_label"] = df["primary_label"].astype(str)
df["filename"] = df["filename"].astype(str)

if USE_SECONDARY_LABELS and "secondary_labels" in df.columns:
    df["secondary_labels_parsed"] = df["secondary_labels"].apply(parse_secondary_labels)
    df["labels"] = df.apply(
        lambda row: list(dict.fromkeys([row["primary_label"]] + row["secondary_labels_parsed"])),
        axis=1
    )
else:
    df["labels"] = df["primary_label"].apply(lambda x: [x])

if SUBSET is not None:
    df = df.sample(min(SUBSET, len(df)), random_state=SEED).reset_index(drop=True)

print("Train size:", len(df))
print(df[["primary_label", "filename", "labels"]].head())

classes = sorted(df["primary_label"].unique().tolist())
class_to_idx = {c: i for i, c in enumerate(classes)}
NUM_CLASSES = len(classes)

print("Num classes:", NUM_CLASSES)

label_counts = df["primary_label"].value_counts()
valid_labels = label_counts[label_counts >= 2].index

df_split = df[df["primary_label"].isin(valid_labels)].copy()
df_single = df[~df["primary_label"].isin(valid_labels)].copy()

train_df, val_df = train_test_split(
    df_split,
    test_size=0.1,
    random_state=SEED,
    stratify=df_split["primary_label"]
)

train_df = pd.concat([train_df, df_single], ignore_index=True).reset_index(drop=True)
val_df = val_df.reset_index(drop=True)

print("Train split:", len(train_df))
print("Val split:", len(val_df))
print("Singleton classes kept only in train:", len(df_single))

class BirdTrainDataset(Dataset):
    def __init__(self, df, audio_dir, class_to_idx, augment=False):
        self.df = df.reset_index(drop=True)
        self.audio_dir = audio_dir
        self.class_to_idx = class_to_idx
        self.augment = augment

    def load_audio(self, path):
        waveform, sr = torchaudio.load(path)

        if waveform.shape[0] > 1:
            waveform = waveform.mean(dim=0, keepdim=True)

        if sr != SR:
            waveform = torchaudio.functional.resample(waveform, sr, SR)

        waveform = waveform.squeeze(0)

        if len(waveform) > AUDIO_LEN:
            start = np.random.randint(0, len(waveform) - AUDIO_LEN + 1)
            waveform = waveform[start:start + AUDIO_LEN]
        else:
            pad = AUDIO_LEN - len(waveform)
            waveform = torch.nn.functional.pad(waveform, (0, pad))

        return waveform.float()

    def make_target(self, labels):
        y = torch.zeros(len(self.class_to_idx), dtype=torch.float32)
        for label in labels:
            if label in self.class_to_idx:
                y[self.class_to_idx[label]] = 1.0
        return y

    def __len__(self):
        return len(self.df)

    def __getitem__(self, idx):
        row = self.df.iloc[idx]
        path = os.path.join(self.audio_dir, row["filename"])

        audio = self.load_audio(path)

        if self.augment:
            if np.random.rand() < 0.5:
                audio = random_gain(audio)
            if np.random.rand() < 0.3:
                audio = add_noise(audio)

        target = self.make_target(row["labels"])
        return audio, target


class MixedBirdDataset(Dataset):

    def __init__(
        self,
        train_df,
        pseudo_rows,
        train_audio_dir,
        pseudo_audio_dir,
        class_to_idx,
        pseudo_ratio=0.25,
        augment=True
    ):
        self.train_df = train_df.reset_index(drop=True)
        self.pseudo_rows = pseudo_rows
        self.train_audio_dir = train_audio_dir
        self.pseudo_audio_dir = pseudo_audio_dir
        self.class_to_idx = class_to_idx
        self.pseudo_ratio = pseudo_ratio
        self.augment = augment


        self.length = max(len(self.train_df), len(self.pseudo_rows), 1)

    def __len__(self):
        return self.length

    def load_audio_segment(self, path, start_sec=None):
        waveform, sr = torchaudio.load(path)

        if waveform.shape[0] > 1:
            waveform = waveform.mean(dim=0, keepdim=True)

        if sr != SR:
            waveform = torchaudio.functional.resample(waveform, sr, SR)

        waveform = waveform.squeeze(0).float()

        if start_sec is None:
            if len(waveform) > AUDIO_LEN:
                start = np.random.randint(0, len(waveform) - AUDIO_LEN + 1)
                waveform = waveform[start:start + AUDIO_LEN]
            else:
                waveform = torch.nn.functional.pad(waveform, (0, AUDIO_LEN - len(waveform)))
        else:
            start = int(start_sec * SR)
            waveform = waveform[start:start + AUDIO_LEN]
            if len(waveform) < AUDIO_LEN:
                waveform = torch.nn.functional.pad(waveform, (0, AUDIO_LEN - len(waveform)))

        return waveform

    def make_hard_target(self, labels):
        y = torch.zeros(len(self.class_to_idx), dtype=torch.float32)
        for label in labels:
            if label in self.class_to_idx:
                y[self.class_to_idx[label]] = 1.0
        return y

    def __getitem__(self, idx):
        use_pseudo = (len(self.pseudo_rows) > 0) and (np.random.rand() < self.pseudo_ratio)

        if use_pseudo:
            row = self.pseudo_rows[np.random.randint(0, len(self.pseudo_rows))]
            path = os.path.join(self.pseudo_audio_dir, row["file_name"])
            audio = self.load_audio_segment(path, start_sec=row["start_sec"])
            target = torch.tensor(row["target"], dtype=torch.float32)
        else:
            row = self.train_df.iloc[np.random.randint(0, len(self.train_df))]
            path = os.path.join(self.train_audio_dir, row["filename"])
            audio = self.load_audio_segment(path, start_sec=None)
            target = self.make_hard_target(row["labels"])

        if self.augment:
            if np.random.rand() < 0.5:
                audio = random_gain(audio)
            if np.random.rand() < 0.3:
                audio = add_noise(audio)

        return audio, target

class ConvBlock1D(nn.Module):
    def __init__(self, in_ch, out_ch, kernel_size, stride):
        super().__init__()
        padding = kernel_size // 2
        self.block = nn.Sequential(
            nn.Conv1d(in_ch, out_ch, kernel_size=kernel_size, stride=stride, padding=padding, bias=False),
            nn.BatchNorm1d(out_ch),
            nn.SiLU(),
        )

    def forward(self, x):
        return self.block(x)


class SEBlock1D(nn.Module):
    def __init__(self, channels, reduction=8):
        super().__init__()
        hidden = max(channels // reduction, 8)
        self.pool = nn.AdaptiveAvgPool1d(1)
        self.fc = nn.Sequential(
            nn.Linear(channels, hidden),
            nn.SiLU(),
            nn.Linear(hidden, channels),
            nn.Sigmoid()
        )

    def forward(self, x):
        b, c, _ = x.shape
        w = self.pool(x).view(b, c)
        w = self.fc(w).view(b, c, 1)
        return x * w


class ResidualBlock1D(nn.Module):
    def __init__(self, channels, kernel_size=5):
        super().__init__()
        padding = kernel_size // 2
        self.conv1 = nn.Conv1d(channels, channels, kernel_size, padding=padding, bias=False)
        self.bn1 = nn.BatchNorm1d(channels)
        self.act = nn.SiLU()
        self.conv2 = nn.Conv1d(channels, channels, kernel_size, padding=padding, bias=False)
        self.bn2 = nn.BatchNorm1d(channels)
        self.se = SEBlock1D(channels)

    def forward(self, x):
        residual = x
        out = self.conv1(x)
        out = self.bn1(out)
        out = self.act(out)

        out = self.conv2(out)
        out = self.bn2(out)
        out = self.se(out)

        out = out + residual
        out = self.act(out)
        return out


class RawBirdCNN(nn.Module):
    def __init__(self, num_classes):
        super().__init__()

        self.stem = nn.Sequential(
            ConvBlock1D(1, 32, kernel_size=11, stride=4),
            ConvBlock1D(32, 64, kernel_size=9, stride=4),
            ConvBlock1D(64, 128, kernel_size=7, stride=4),
        )

        self.stage1 = nn.Sequential(
            ResidualBlock1D(128),
            nn.MaxPool1d(4),
        )

        self.stage2 = nn.Sequential(
            ConvBlock1D(128, 192, kernel_size=5, stride=2),
            ResidualBlock1D(192),
            nn.MaxPool1d(2),
        )

        self.stage3 = nn.Sequential(
            ConvBlock1D(192, 256, kernel_size=5, stride=2),
            ResidualBlock1D(256),
        )

        self.pool_avg = nn.AdaptiveAvgPool1d(1)
        self.pool_max = nn.AdaptiveMaxPool1d(1)

        self.head = nn.Sequential(
            nn.Linear(256 * 2, 256),
            nn.SiLU(),
            nn.Dropout(0.2),
            nn.Linear(256, num_classes)
        )

    def forward(self, x):
        x = x.unsqueeze(1)

        x = self.stem(x)
        x = self.stage1(x)
        x = self.stage2(x)
        x = self.stage3(x)

        x_avg = self.pool_avg(x).squeeze(-1)
        x_max = self.pool_max(x).squeeze(-1)
        x = torch.cat([x_avg, x_max], dim=1)

        logits = self.head(x)
        return logits


train_dataset = BirdTrainDataset(train_df, TRAIN_AUDIO_DIR, class_to_idx, augment=True)
val_dataset = BirdTrainDataset(val_df, TRAIN_AUDIO_DIR, class_to_idx, augment=False)

train_loader = DataLoader(
    train_dataset,
    batch_size=BATCH_SIZE,
    shuffle=True,
    num_workers=NUM_WORKERS,
    pin_memory=False,
    drop_last=True
)

val_loader = DataLoader(
    val_dataset,
    batch_size=BATCH_SIZE,
    shuffle=False,
    num_workers=NUM_WORKERS,
    pin_memory=False
)


class_counts = np.zeros(NUM_CLASSES, dtype=np.float32)
for labels in train_df["labels"]:
    y = np.zeros(NUM_CLASSES, dtype=np.float32)
    for label in labels:
        if label in class_to_idx:
            y[class_to_idx[label]] = 1.0
    class_counts += y

pos_weight = (len(train_df) - class_counts) / (class_counts + 1e-6)
pos_weight = np.clip(pos_weight, 1.0, 20.0)
pos_weight = torch.tensor(pos_weight, dtype=torch.float32, device=DEVICE)

criterion = nn.BCEWithLogitsLoss(pos_weight=pos_weight)

def train_one_epoch(model, loader, optimizer, scaler):
    model.train()
    total_loss = 0.0

    for x, y in loader:
        x = x.to(DEVICE, non_blocking=False)
        y = y.to(DEVICE, non_blocking=False)

        optimizer.zero_grad(set_to_none=True)

        with torch.amp.autocast("cuda" if DEVICE=="cuda" else "cpu", enabled=(DEVICE == "cuda")):
            logits = model(x)
            loss = criterion(logits, y)

        if DEVICE == "cuda":
            scaler.scale(loss).backward()
            scaler.step(optimizer)
            scaler.update()
        else:
            loss.backward()
            optimizer.step()

        total_loss += loss.item()
        
        # --- MAC MPS MEMORY LEAK FIX ---
        del x, y, logits, loss
        if DEVICE == "mps":
            torch.mps.empty_cache()
        # -------------------------------

    return total_loss / len(loader)


@torch.no_grad()
def validate(model, loader):
    model.eval()
    total_loss = 0.0
    all_targets = []
    all_probs = []

    for x, y in loader:
        x = x.to(DEVICE, non_blocking=False)
        y = y.to(DEVICE, non_blocking=False)

        with torch.amp.autocast("cuda" if DEVICE=="cuda" else "cpu", enabled=(DEVICE == "cuda")):
            logits = model(x)
            loss = criterion(logits, y)

        probs = torch.sigmoid(logits)

        total_loss += loss.item()
        all_targets.append(y.cpu().numpy())
        all_probs.append(probs.cpu().numpy())
        
        # --- MAC MPS MEMORY LEAK FIX ---
        del x, y, logits, loss, probs
        if DEVICE == "mps":
            torch.mps.empty_cache()
        # -------------------------------

    all_targets = np.concatenate(all_targets, axis=0)
    all_probs = np.concatenate(all_probs, axis=0)

    auc = macro_roc_auc(all_targets, all_probs)
    return total_loss / len(loader), auc

def run_training(model, train_loader, val_loader, epochs, lr, save_path, save_name):
    optimizer = torch.optim.AdamW(model.parameters(), lr=lr, weight_decay=1e-4)
    scheduler = torch.optim.lr_scheduler.CosineAnnealingLR(optimizer, T_max=epochs)
    scaler = torch.amp.GradScaler("cuda", enabled=(DEVICE == "cuda"))

    best_auc = -1.0

    for epoch in range(epochs):
        start_time = time.time()

        train_loss = train_one_epoch(model, train_loader, optimizer, scaler)
        val_loss, val_auc = validate(model, val_loader)
        scheduler.step()

        elapsed = time.time() - start_time

        print(
            f"[{save_name}] Epoch {epoch+1}/{epochs} | "
            f"train_loss={train_loss:.4f} | "
            f"val_loss={val_loss:.4f} | "
            f"val_auc={val_auc:.4f} | "
            f"time={format_seconds(elapsed)}"
        )

        if val_auc > best_auc:
            best_auc = val_auc
            torch.save(
                {
                    "model_state_dict": model.state_dict(),
                    "classes": classes,
                    "sample_rate": SR,
                    "duration": DURATION,
                    "model_name": "RawBirdCNN",
                },
                save_path
            )
            print(f"Saved best {save_name} model.")
            
        import gc
        gc.collect()

    print(f"Best {save_name} val AUC: {best_auc:.6f}")
    return best_auc

baseline_model = RawBirdCNN(NUM_CLASSES).to(DEVICE)

baseline_best_auc = run_training(
    model=baseline_model,
    train_loader=train_loader,
    val_loader=val_loader,
    epochs=BASELINE_EPOCHS,
    lr=BASELINE_LR,
    save_path=BASELINE_MODEL_PATH,
    save_name="baseline"
)

baseline_ckpt = torch.load(BASELINE_MODEL_PATH, map_location=DEVICE)
baseline_model.load_state_dict(baseline_ckpt["model_state_dict"])
baseline_model.eval()


Train size: 35549
  primary_label                 filename     labels
0       1161364  1161364/iNat1216197.ogg  [1161364]
1       1161364  1161364/iNat1114648.ogg  [1161364]
2       1161364   1161364/iNat810195.ogg  [1161364]
3       1161364   1161364/iNat818781.ogg  [1161364]
4       1161364   1161364/iNat556514.ogg  [1161364]
Num classes: 206
Train split: 31994
Val split: 3555
Singleton classes kept only in train: 4
[baseline] Epoch 1/6 | train_loss=0.2814 | val_loss=0.2293 | val_auc=0.8154 | time=10:04
Saved best baseline model.
[baseline] Epoch 2/6 | train_loss=0.2169 | val_loss=0.1996 | val_auc=0.8746 | time=10:11
Saved best baseline model.
[baseline] Epoch 3/6 | train_loss=0.1886 | val_loss=0.1771 | val_auc=0.9022 | time=10:00
Saved best baseline model.
[baseline] Epoch 4/6 | train_loss=0.1706 | val_loss=0.1629 | val_auc=0.9151 | time=10:24
Saved best baseline model.
[baseline] Epoch 5/6 | train_loss=0.1578 | val_loss=0.1535 | val_auc=0.9294 | time=10:25
Saved best baseline model

RawBirdCNN(
  (stem): Sequential(
    (0): ConvBlock1D(
      (block): Sequential(
        (0): Conv1d(1, 32, kernel_size=(11,), stride=(4,), padding=(5,), bias=False)
        (1): BatchNorm1d(32, eps=1e-05, momentum=0.1, affine=True, track_running_stats=True)
        (2): SiLU()
      )
    )
    (1): ConvBlock1D(
      (block): Sequential(
        (0): Conv1d(32, 64, kernel_size=(9,), stride=(4,), padding=(4,), bias=False)
        (1): BatchNorm1d(64, eps=1e-05, momentum=0.1, affine=True, track_running_stats=True)
        (2): SiLU()
      )
    )
    (2): ConvBlock1D(
      (block): Sequential(
        (0): Conv1d(64, 128, kernel_size=(7,), stride=(4,), padding=(3,), bias=False)
        (1): BatchNorm1d(128, eps=1e-05, momentum=0.1, affine=True, track_running_stats=True)
        (2): SiLU()
      )
    )
  )
  (stage1): Sequential(
    (0): ResidualBlock1D(
      (conv1): Conv1d(128, 128, kernel_size=(5,), stride=(1,), padding=(2,), bias=False)
      (bn1): BatchNorm1d(128, eps=1e-0

In [9]:
TRAIN_SOUNDSCAPES_DIR = "birdclef-2026/train_soundscapes"
MODEL_PATH = "raw_bird_base_best.pt"
PSEUDO_JSON_PATH = "pseudo_labels_from_train_soundscapes.json"
PSEUDO_MODEL_PATH = "raw_bird_pseudo_best.pt"

DEVICE = "cuda" if torch.cuda.is_available() else "mps" if torch.backends.mps.is_available() else "cpu"

USE_SECONDARY_LABELS = False
SUBSET = None

BATCH_SIZE = 32
NUM_WORKERS = 0 # Changed from 2 to 0 for macOS


PSEUDO_CONF_THRESHOLD = 0.90
PSEUDO_MIN_KEEP_PROB = 0.10
PSEUDO_BATCH_SIZE = 32

PSEUDO_RATIO = 0.25
PSEUDO_FINETUNE_EPOCHS = 4
PSEUDO_FINETUNE_LR = 5e-4


seed_everything(SEED)



def macro_roc_auc(y_true, y_pred):
    scores = []
    for i in range(y_true.shape[1]):
        if len(np.unique(y_true[:, i])) < 2:
            continue
        try:
            scores.append(roc_auc_score(y_true[:, i], y_pred[:, i]))
        except Exception:
            pass
    if len(scores) == 0:
        return np.nan
    return float(np.mean(scores))

def format_seconds(sec):
    m = int(sec // 60)
    s = int(sec % 60)
    return f"{m:02d}:{s:02d}"


df = pd.read_csv(TRAIN_CSV)
df["primary_label"] = df["primary_label"].astype(str)
df["filename"] = df["filename"].astype(str)

if USE_SECONDARY_LABELS and "secondary_labels" in df.columns:
    df["secondary_labels_parsed"] = df["secondary_labels"].apply(parse_secondary_labels)
    df["labels"] = df.apply(
        lambda row: list(dict.fromkeys([row["primary_label"]] + row["secondary_labels_parsed"])),
        axis=1
    )
else:
    df["labels"] = df["primary_label"].apply(lambda x: [x])

if SUBSET is not None:
    df = df.sample(min(SUBSET, len(df)), random_state=SEED).reset_index(drop=True)

print("Train size:", len(df))
print(df[["primary_label", "filename", "labels"]].head())

classes = sorted(df["primary_label"].unique().tolist())
class_to_idx = {c: i for i, c in enumerate(classes)}
NUM_CLASSES = len(classes)

print("Num classes:", NUM_CLASSES)


label_counts = df["primary_label"].value_counts()
valid_labels = label_counts[label_counts >= 2].index

df_split = df[df["primary_label"].isin(valid_labels)].copy()
df_single = df[~df["primary_label"].isin(valid_labels)].copy()

train_df, val_df = train_test_split(
    df_split,
    test_size=0.1,
    random_state=SEED,
    stratify=df_split["primary_label"]
)

train_df = pd.concat([train_df, df_single], ignore_index=True).reset_index(drop=True)
val_df = val_df.reset_index(drop=True)

print("Train split:", len(train_df))
print("Val split:", len(val_df))
print("Singleton classes kept only in train:", len(df_single))


class BirdTrainDataset(Dataset):
    def __init__(self, df, audio_dir, class_to_idx, augment=False):
        self.df = df.reset_index(drop=True)
        self.audio_dir = audio_dir
        self.class_to_idx = class_to_idx
        self.augment = augment

    def load_audio(self, path):
        waveform, sr = torchaudio.load(path)

        if waveform.shape[0] > 1:
            waveform = waveform.mean(dim=0, keepdim=True)

        if sr != SR:
            waveform = torchaudio.functional.resample(waveform, sr, SR)

        waveform = waveform.squeeze(0)

        if len(waveform) > AUDIO_LEN:
            start = np.random.randint(0, len(waveform) - AUDIO_LEN + 1)
            waveform = waveform[start:start + AUDIO_LEN]
        else:
            pad = AUDIO_LEN - len(waveform)
            waveform = torch.nn.functional.pad(waveform, (0, pad))

        return waveform.float()

    def make_target(self, labels):
        y = torch.zeros(len(self.class_to_idx), dtype=torch.float32)
        for label in labels:
            if label in self.class_to_idx:
                y[self.class_to_idx[label]] = 1.0
        return y

    def __len__(self):
        return len(self.df)

    def __getitem__(self, idx):
        row = self.df.iloc[idx]
        path = os.path.join(self.audio_dir, row["filename"])

        audio = self.load_audio(path)

        if self.augment:
            if np.random.rand() < 0.5:
                audio = random_gain(audio)
            if np.random.rand() < 0.3:
                audio = add_noise(audio)

        target = self.make_target(row["labels"])
        return audio, target

class MixedBirdDataset(Dataset):
    def __init__(
        self,
        train_df,
        pseudo_rows,
        train_audio_dir,
        pseudo_audio_dir,
        class_to_idx,
        pseudo_ratio=0.25,
        augment=True
    ):
        self.train_df = train_df.reset_index(drop=True)
        self.pseudo_rows = pseudo_rows
        self.train_audio_dir = train_audio_dir
        self.pseudo_audio_dir = pseudo_audio_dir
        self.class_to_idx = class_to_idx
        self.pseudo_ratio = pseudo_ratio
        self.augment = augment
        self.length = max(len(self.train_df), len(self.pseudo_rows), 1)

    def __len__(self):
        return self.length

    def load_audio_segment(self, path, start_sec=None):
        waveform, sr = torchaudio.load(path)

        if waveform.shape[0] > 1:
            waveform = waveform.mean(dim=0, keepdim=True)

        if sr != SR:
            waveform = torchaudio.functional.resample(waveform, sr, SR)

        waveform = waveform.squeeze(0).float()

        if start_sec is None:
            if len(waveform) > AUDIO_LEN:
                start = np.random.randint(0, len(waveform) - AUDIO_LEN + 1)
                waveform = waveform[start:start + AUDIO_LEN]
            else:
                waveform = torch.nn.functional.pad(waveform, (0, AUDIO_LEN - len(waveform)))
        else:
            start = int(start_sec * SR)
            waveform = waveform[start:start + AUDIO_LEN]
            if len(waveform) < AUDIO_LEN:
                waveform = torch.nn.functional.pad(waveform, (0, AUDIO_LEN - len(waveform)))

        return waveform

    def make_hard_target(self, labels):
        y = torch.zeros(len(self.class_to_idx), dtype=torch.float32)
        for label in labels:
            if label in self.class_to_idx:
                y[self.class_to_idx[label]] = 1.0
        return y

    def __getitem__(self, idx):
        use_pseudo = (len(self.pseudo_rows) > 0) and (np.random.rand() < self.pseudo_ratio)

        if use_pseudo:
            row = self.pseudo_rows[np.random.randint(0, len(self.pseudo_rows))]
            path = os.path.join(self.pseudo_audio_dir, row["file_name"])
            audio = self.load_audio_segment(path, start_sec=row["start_sec"])
            target = torch.tensor(row["target"], dtype=torch.float32)
        else:
            row = self.train_df.iloc[np.random.randint(0, len(self.train_df))]
            path = os.path.join(self.train_audio_dir, row["filename"])
            audio = self.load_audio_segment(path, start_sec=None)
            target = self.make_hard_target(row["labels"])

        if self.augment:
            if np.random.rand() < 0.5:
                audio = random_gain(audio)
            if np.random.rand() < 0.3:
                audio = add_noise(audio)

        return audio, target


class ConvBlock1D(nn.Module):
    def __init__(self, in_ch, out_ch, kernel_size, stride):
        super().__init__()
        padding = kernel_size // 2
        self.block = nn.Sequential(
            nn.Conv1d(in_ch, out_ch, kernel_size=kernel_size, stride=stride, padding=padding, bias=False),
            nn.BatchNorm1d(out_ch),
            nn.SiLU(),
        )

    def forward(self, x):
        return self.block(x)

class SEBlock1D(nn.Module):
    def __init__(self, channels, reduction=8):
        super().__init__()
        hidden = max(channels // reduction, 8)
        self.pool = nn.AdaptiveAvgPool1d(1)
        self.fc = nn.Sequential(
            nn.Linear(channels, hidden),
            nn.SiLU(),
            nn.Linear(hidden, channels),
            nn.Sigmoid()
        )

    def forward(self, x):
        b, c, _ = x.shape
        w = self.pool(x).view(b, c)
        w = self.fc(w).view(b, c, 1)
        return x * w

class ResidualBlock1D(nn.Module):
    def __init__(self, channels, kernel_size=5):
        super().__init__()
        padding = kernel_size // 2
        self.conv1 = nn.Conv1d(channels, channels, kernel_size, padding=padding, bias=False)
        self.bn1 = nn.BatchNorm1d(channels)
        self.act = nn.SiLU()
        self.conv2 = nn.Conv1d(channels, channels, kernel_size, padding=padding, bias=False)
        self.bn2 = nn.BatchNorm1d(channels)
        self.se = SEBlock1D(channels)

    def forward(self, x):
        residual = x
        out = self.conv1(x)
        out = self.bn1(out)
        out = self.act(out)
        out = self.conv2(out)
        out = self.bn2(out)
        out = self.se(out)
        out = out + residual
        out = self.act(out)
        return out

class RawBirdCNN(nn.Module):
    def __init__(self, num_classes):
        super().__init__()

        self.stem = nn.Sequential(
            ConvBlock1D(1, 32, kernel_size=11, stride=4),
            ConvBlock1D(32, 64, kernel_size=9, stride=4),
            ConvBlock1D(64, 128, kernel_size=7, stride=4),
        )

        self.stage1 = nn.Sequential(
            ResidualBlock1D(128),
            nn.MaxPool1d(4),
        )

        self.stage2 = nn.Sequential(
            ConvBlock1D(128, 192, kernel_size=5, stride=2),
            ResidualBlock1D(192),
            nn.MaxPool1d(2),
        )

        self.stage3 = nn.Sequential(
            ConvBlock1D(192, 256, kernel_size=5, stride=2),
            ResidualBlock1D(256),
        )

        self.pool_avg = nn.AdaptiveAvgPool1d(1)
        self.pool_max = nn.AdaptiveMaxPool1d(1)

        self.head = nn.Sequential(
            nn.Linear(256 * 2, 256),
            nn.SiLU(),
            nn.Dropout(0.2),
            nn.Linear(256, num_classes)
        )

    def forward(self, x):
        x = x.unsqueeze(1)
        x = self.stem(x)
        x = self.stage1(x)
        x = self.stage2(x)
        x = self.stage3(x)

        x_avg = self.pool_avg(x).squeeze(-1)
        x_max = self.pool_max(x).squeeze(-1)
        x = torch.cat([x_avg, x_max], dim=1)

        logits = self.head(x)
        return logits


val_dataset = BirdTrainDataset(val_df, TRAIN_AUDIO_DIR, class_to_idx, augment=False)
val_loader = DataLoader(
    val_dataset,
    batch_size=BATCH_SIZE,
    shuffle=False,
    num_workers=NUM_WORKERS,
    pin_memory=False
)

class_counts = np.zeros(NUM_CLASSES, dtype=np.float32)
for labels in train_df["labels"]:
    y = np.zeros(NUM_CLASSES, dtype=np.float32)
    for label in labels:
        if label in class_to_idx:
            y[class_to_idx[label]] = 1.0
    class_counts += y

pos_weight = (len(train_df) - class_counts) / (class_counts + 1e-6)
pos_weight = np.clip(pos_weight, 1.0, 20.0)
pos_weight = torch.tensor(pos_weight, dtype=torch.float32, device=DEVICE)

criterion = nn.BCEWithLogitsLoss(pos_weight=pos_weight)


@torch.no_grad()
def validate(model, loader):
    model.eval()
    total_loss = 0.0
    all_targets = []
    all_probs = []

    for x, y in loader:
        x = x.to(DEVICE, non_blocking=False)
        y = y.to(DEVICE, non_blocking=False)

        with torch.amp.autocast("cuda" if DEVICE=="cuda" else "cpu", enabled=(DEVICE == "cuda")):
            logits = model(x)
            loss = criterion(logits, y)

        probs = torch.sigmoid(logits)

        total_loss += loss.item()
        all_targets.append(y.cpu().numpy())
        all_probs.append(probs.cpu().numpy())

        # --- MAC MPS MEMORY LEAK FIX ---
        del x, y, logits, loss, probs
        if DEVICE == "mps":
            torch.mps.empty_cache()
        # -------------------------------

    all_targets = np.concatenate(all_targets, axis=0)
    all_probs = np.concatenate(all_probs, axis=0)
    auc = macro_roc_auc(all_targets, all_probs)

    return total_loss / len(loader), auc


model = RawBirdCNN(NUM_CLASSES).to(DEVICE)

ckpt = torch.load(MODEL_PATH, map_location=DEVICE)
if isinstance(ckpt, dict) and "model_state_dict" in ckpt:
    model.load_state_dict(ckpt["model_state_dict"])
else:
    model.load_state_dict(ckpt)

model.eval()
print("Loaded baseline model from:", MODEL_PATH)

base_val_loss, base_val_auc = validate(model, val_loader)
print(f"Baseline val_loss={base_val_loss:.4f} | val_auc={base_val_auc:.4f}")


def load_audio_full(path):
    waveform, sr = torchaudio.load(path)
    if waveform.shape[0] > 1:
        waveform = waveform.mean(dim=0, keepdim=True)
    if sr != SR:
        waveform = torchaudio.functional.resample(waveform, sr, SR)
    return waveform.squeeze(0).float()

def make_chunks(waveform, chunk_len=AUDIO_LEN):
    chunks, starts = [], []
    for start in range(0, len(waveform), chunk_len):
        chunk = waveform[start:start + chunk_len]
        if len(chunk) < chunk_len:
            chunk = torch.nn.functional.pad(chunk, (0, chunk_len - len(chunk)))
        chunks.append(chunk)
        starts.append(start // SR)
    return chunks, starts

@torch.no_grad()
def predict_chunks(model, chunks, batch_size=32):
    preds = []
    model.eval()

    for i in range(0, len(chunks), batch_size):
        batch = torch.stack(chunks[i:i + batch_size]).to(DEVICE)
        with torch.amp.autocast("cuda" if DEVICE=="cuda" else "cpu", enabled=(DEVICE == "cuda")):
            logits = model(batch)
            probs = torch.sigmoid(logits)
        preds.append(probs.cpu().numpy())

        # --- MAC MPS MEMORY LEAK FIX ---
        del batch, logits, probs
        if DEVICE == "mps":
            torch.mps.empty_cache()
        # -------------------------------

    return np.concatenate(preds, axis=0)

if not os.path.exists(TRAIN_SOUNDSCAPES_DIR):
    # Skip if missing
    soundscape_files = []
else:
    soundscape_files = sorted([
        f for f in os.listdir(TRAIN_SOUNDSCAPES_DIR)
        if f.endswith(".ogg")
    ])

print("Found train soundscapes for pseudo-labeling:", len(soundscape_files))

pseudo_rows = []

for file_name in tqdm(soundscape_files, desc="Pseudo-labeling train_soundscapes"):
    path = os.path.join(TRAIN_SOUNDSCAPES_DIR, file_name)
    waveform = load_audio_full(path)
    chunks, starts = make_chunks(waveform)
    
    if len(chunks) > 0:
        probs = predict_chunks(model, chunks, batch_size=PSEUDO_BATCH_SIZE)

        for start_sec, prob_vec in zip(starts, probs):
            max_prob = float(prob_vec.max())
            if max_prob < PSEUDO_CONF_THRESHOLD:
                continue

            prob_vec = prob_vec.copy()
            prob_vec[prob_vec < PSEUDO_MIN_KEEP_PROB] = 0.0

            pseudo_rows.append({
                "file_name": file_name,
                "start_sec": int(start_sec),
                "target": prob_vec.tolist(),
                "max_prob": max_prob
            })
            
    import gc
    gc.collect()

print("Selected pseudo-labeled chunks:", len(pseudo_rows))

if len(pseudo_rows) > 0:
    with open(PSEUDO_JSON_PATH, "w") as f:
        json.dump(pseudo_rows, f)
    print("Saved pseudo labels to:", PSEUDO_JSON_PATH)


mixed_train_dataset = MixedBirdDataset(
    train_df=train_df,
    pseudo_rows=pseudo_rows,
    train_audio_dir=TRAIN_AUDIO_DIR,
    pseudo_audio_dir=TRAIN_SOUNDSCAPES_DIR,
    class_to_idx=class_to_idx,
    pseudo_ratio=PSEUDO_RATIO,
    augment=True
)

mixed_train_loader = DataLoader(
    mixed_train_dataset,
    batch_size=BATCH_SIZE,
    shuffle=True,
    num_workers=NUM_WORKERS,
    pin_memory=False,
    drop_last=True
)

pseudo_model = RawBirdCNN(NUM_CLASSES).to(DEVICE)
pseudo_model.load_state_dict(model.state_dict())

optimizer = torch.optim.AdamW(pseudo_model.parameters(), lr=PSEUDO_FINETUNE_LR, weight_decay=1e-4)
scheduler = torch.optim.lr_scheduler.CosineAnnealingLR(optimizer, T_max=PSEUDO_FINETUNE_EPOCHS)
scaler = torch.amp.GradScaler("cuda", enabled=(DEVICE == "cuda"))

def train_one_epoch(model, loader, optimizer, scaler):
    model.train()
    total_loss = 0.0

    for x, y in loader:
        x = x.to(DEVICE, non_blocking=False)
        y = y.to(DEVICE, non_blocking=False)

        optimizer.zero_grad(set_to_none=True)

        with torch.amp.autocast("cuda" if DEVICE=="cuda" else "cpu", enabled=(DEVICE == "cuda")):
            logits = model(x)
            loss = criterion(logits, y)

        if DEVICE == "cuda":
            scaler.scale(loss).backward()
            scaler.step(optimizer)
            scaler.update()
        else:
            loss.backward()
            optimizer.step()

        total_loss += loss.item()

        # --- MAC MPS MEMORY LEAK FIX ---
        del x, y, logits, loss
        if DEVICE == "mps":
            torch.mps.empty_cache()
        # -------------------------------

    return total_loss / len(loader)

best_auc = -1.0

for epoch in range(PSEUDO_FINETUNE_EPOCHS):
    start_time = time.time()

    train_loss = train_one_epoch(pseudo_model, mixed_train_loader, optimizer, scaler)
    val_loss, val_auc = validate(pseudo_model, val_loader)
    scheduler.step()

    elapsed = time.time() - start_time

    print(
        f"[pseudo] Epoch {epoch+1}/{PSEUDO_FINETUNE_EPOCHS} | "
        f"train_loss={train_loss:.4f} | "
        f"val_loss={val_loss:.4f} | "
        f"val_auc={val_auc:.4f} | "
        f"time={format_seconds(elapsed)}"
    )

    if val_auc > best_auc:
        best_auc = val_auc
        torch.save(
            {
                "model_state_dict": pseudo_model.state_dict(),
                "classes": classes,
                "sample_rate": SR,
                "duration": DURATION,
                "model_name": "RawBirdCNN_pseudo_train_soundscapes",
            },
            PSEUDO_MODEL_PATH
        )
        print("Saved best pseudo model.")
    
    import gc
    gc.collect()

print("Best pseudo val AUC:", best_auc)
print("Saved pseudo model to:", PSEUDO_MODEL_PATH)


Train size: 35549
  primary_label                 filename     labels
0       1161364  1161364/iNat1216197.ogg  [1161364]
1       1161364  1161364/iNat1114648.ogg  [1161364]
2       1161364   1161364/iNat810195.ogg  [1161364]
3       1161364   1161364/iNat818781.ogg  [1161364]
4       1161364   1161364/iNat556514.ogg  [1161364]
Num classes: 206
Train split: 31994
Val split: 3555
Singleton classes kept only in train: 4
Loaded baseline model from: raw_bird_base_best.pt
Baseline val_loss=0.1483 | val_auc=0.9308
Found train soundscapes for pseudo-labeling: 10658


Pseudo-labeling train_soundscapes:   0%|          | 0/10658 [00:00<?, ?it/s]

Selected pseudo-labeled chunks: 11916
Saved pseudo labels to: pseudo_labels_from_train_soundscapes.json
[pseudo] Epoch 1/4 | train_loss=0.1915 | val_loss=0.1648 | val_auc=0.9155 | time=10:23
Saved best pseudo model.
[pseudo] Epoch 2/4 | train_loss=0.1781 | val_loss=0.1549 | val_auc=0.9263 | time=10:22
Saved best pseudo model.
[pseudo] Epoch 3/4 | train_loss=0.1708 | val_loss=0.1519 | val_auc=0.9299 | time=10:34
Saved best pseudo model.
[pseudo] Epoch 4/4 | train_loss=0.1627 | val_loss=0.1452 | val_auc=0.9323 | time=10:33
Saved best pseudo model.
Best pseudo val AUC: 0.9323461445798248
Saved pseudo model to: raw_bird_pseudo_best.pt


In [12]:
import torch
import numpy as np
import os
from tqdm.auto import tqdm

def evaluate_accuracy(model, model_path, val_loader, device):
    print(f"Evaluating {model_path} for Accuracy...")
    if not os.path.exists(model_path):
        print(f"Model weights not found at {model_path}")
        return
        
    ckpt = torch.load(model_path, map_location=device)
    if isinstance(ckpt, dict) and "model_state_dict" in ckpt:
        model.load_state_dict(ckpt["model_state_dict"])
    else:
        model.load_state_dict(ckpt)
        
    model.eval()
    
    all_targets = []
    all_probs = []

    with torch.no_grad():
        for x, y in tqdm(val_loader, desc="Evaluating"):
            x = x.to(device, non_blocking=False)
            
            # Match the training setup amp autocast
            with torch.amp.autocast("cuda" if device=="cuda" else "cpu", enabled=(device == "cuda")):
                logits = model(x)
                
            probs = torch.sigmoid(logits)
            
            all_targets.append(y.cpu().numpy())
            all_probs.append(probs.cpu().numpy())
            
            if device == "mps":
                del x, logits, probs
                torch.mps.empty_cache()

    # Concatenate all batches to get the full validation set
    all_targets = np.concatenate(all_targets, axis=0)
    all_probs = np.concatenate(all_probs, axis=0)

    # Get the index of the highest probability prediction for each sample
    preds_top1 = np.argmax(all_probs, axis=1)
    
    # Check if the top predicted class is one of the true labels (value == 1.0)
    correct_predictions = all_targets[np.arange(len(all_targets)), preds_top1] == 1.0
    
    # Average across the entire validation set
    acc = correct_predictions.mean()
    
    print(f"Validation Average Top-1 Accuracy: {acc:.4f}\n")
    return acc

eval_model = RawBirdCNN(NUM_CLASSES).to(DEVICE)

# Evaluate Baseline Accuracy
evaluate_accuracy(eval_model, "raw_bird_cnn_best.pt", val_loader, DEVICE)

# Evaluate Finetuned (Pseudo) Accuracy
evaluate_accuracy(eval_model, "raw_bird_pseudo_best.pt", val_loader, DEVICE)

Evaluating raw_bird_cnn_best.pt for Accuracy...


Evaluating:   0%|          | 0/112 [00:00<?, ?it/s]

Validation Average Top-1 Accuracy: 0.3865

Evaluating raw_bird_pseudo_best.pt for Accuracy...


Evaluating:   0%|          | 0/112 [00:00<?, ?it/s]

Validation Average Top-1 Accuracy: 0.4214



np.float64(0.4213783403656821)